# NPU 实践

恭喜你完成了 CANN 基础知识课程的全部 3 节课！本节通过四道难度递进的实践题，在 NPU 上动手验证所学知识。

| 课程 | 核心问题 | 一句话回答 |
|:---:|:---|:---|
| ① 人工智能基础 | AI 在算什么？ | 模型 = 参数化函数 $y=f(x;\theta)$，训练求 $\theta$，推理用 $\theta$，计算图把算子串成网络 |
| ② 什么是 NPU | 硬件怎么算？ | DaVinci 架构，AI Core = Cube + Vector + Scalar，多核并行 + Tiling 切分 |
| ③ 什么是 CANN | 软件怎么管？ | 翻译官 + 调度员 + 工具箱，从框架适配到驱动，九层组件协同 |

---
## 实践须知

每道题的代码单元中已标注 `# TODO` 的部分需要你完成。完成后运行验证单元查看批改结果。

**关键 API 速查**：

| 操作 | API | 说明 |
|:---|:---|:---|
| 搬到 NPU | `tensor.npu()` | Host → Device |
| 搬回 CPU | `tensor.cpu()` | Device → Host |
| NPU 同步 | `torch.npu.synchronize()` | 等待异步计算完成 |
| 矩阵乘法 | `torch.matmul(a, b)` | 自动调用 CANN MatMul 算子 |
| 卷积 | `F.conv2d(x, w, padding=..., groups=...)` | 自动调用 CANN Conv 算子 |

---
## Hello NPU：你的第一个 NPU 程序

在开始实践题之前，先用 3 行代码跑通 NPU，确认环境就绪。

核心流程：`import torch_npu` → `.npu()` 搬到设备 → `.cpu()` 搬回验证。

### 环境初始化

> CANNLab 在线环境有时未自动加载 CANN 环境变量，需要手动 `source set_env.sh`。
> 本地环境若已配置好 CANN，此 cell 可跳过。

In [ ]:
import os, subprocess
for p in [os.path.join(os.environ.get('ASCEND_TOOLKIT_HOME', ''), 'set_env.sh'),
          '/home/developer/Ascend/cann/set_env.sh',
          '/usr/local/Ascend/ascend-toolkit/set_env.sh']:
    if p and os.path.exists(p):
        for line in subprocess.run(['bash', '-c', f'source {p} && env'], capture_output=True, text=True).stdout.splitlines():
            if '=' in line: k, v = line.split('=', 1); os.environ[k] = v
        print(f'CANN loaded: {p}')
        break

In [ ]:
import warnings; warnings.filterwarnings('ignore', message=r'.*(owner does not match|Permission mismatch|TASK_QUEUE_ENABLE)')
import torch
import torch_npu

# ① 检查 NPU 是否可用
print(f"NPU 可用: {torch.npu.is_available()}")
print(f"NPU 设备: {torch.npu.get_device_name(0)}")

# ② 在 NPU 上做一个简单加法
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
c = (a.npu() + b.npu()).cpu()  # 搬到 NPU 计算，再搬回 CPU

print(f"a = {a.tolist()}")
print(f"b = {b.tolist()}")
print(f"a + b = {c.tolist()}  (Hello, we are running on NPU.)")

---
## 一、实践题

下面四道实践题难度递进，请在代码单元中完成作答后运行验证。

| 题号 | 难度 | 考察点 | 目标 |
|:---:|:---:|:---|:---|
| 第 1 题 | ★☆☆ 基础 | 张量创建、设备搬运、逐元素运算 | 在 NPU 上完成 $z = x^2 + y$ 并验证 |
| 第 2 题 | ★★☆ 进阶 | 矩阵乘法、性能对比、数据类型 | 对比 CPU/NPU 在不同规模下的矩阵乘法耗时 |
| 第 3 题 | ★★★ 挑战 | Batch 运算、算子调用开销、性能分析 | 对比 `torch.bmm` 与循环 `torch.matmul` 的性能差异，观察 batch 大小的影响 |
| 第 4 题 | ★★★ 挑战 | 卷积运算、图像处理、算术强度 | 在 NPU 上加速图像高斯模糊，对比 CPU/NPU 卷积性能 |

### 第 1 题（基础 ★☆☆）：在 NPU 上计算 $z = x^2 + y$

**要求**：
1. 创建两个形状为 `(1000,)` 的 FP32 张量 `x` 和 `y`，`x` 的值全为 2.0，`y` 的值全为 3.0
2. 将它们搬到 NPU
3. 在 NPU 上计算 `z = x² + y`（提示：可以用 `torch.pow(x, 2)` 或 `x * x`）
4. 将结果搬回 CPU 并验证所有元素等于 7.0

**预期结果**：`z` 的所有元素均为 $2.0^2 + 3.0 = 7.0$

**API 速查**：

| 操作 | API | 示例 |
|:---|:---|:---|
| 创建全量张量 | `torch.full(shape, value, dtype=...)` | `torch.full((1000,), 2.0, dtype=torch.float32)` |
| 搬到 NPU | `tensor.npu()` | `x_npu = x.npu()` |
| 搬回 CPU | `tensor.cpu()` | `z = z_npu.cpu()` |
| 逐元素平方 | `x * x` 或 `torch.pow(x, 2)` | `sq = x_npu * x_npu` |
| 逐元素加法 | `a + b` | `z_npu = sq + y_npu` |
| 验证相等 | `torch.allclose(a, b)` | `torch.allclose(z, expected)` |

In [ ]:
import torch
import torch_npu

# ========== 第 1 题：在 NPU 上计算 z = x² + y ==========
# TODO: 在下方完成你的代码

# Step 1: 创建 x 和 y（CPU 上，FP32，形状 (1000,)，x 全为 2.0，y 全为 3.0）
x = 
y = 

# Step 2: 搬到 NPU
x_npu = 
y_npu = 

# Step 3: 在 NPU 上计算 z = x² + y
z_npu = 

# Step 4: 搬回 CPU
z = 

# 验证
expected = torch.full((1000,), 7.0)
print(f"z 的前 5 个元素: {z[:5].tolist()}")
print(f"z 的设备: {z_npu.device}")
print(f"验证结果: {'✓ 通过' if torch.allclose(z, expected) else '✗ 失败'}")

### 第 2 题（进阶 ★★☆）：CPU vs NPU 矩阵乘法性能对比

**要求**：
1. 对矩阵规模 `N = [128, 512, 2048]`，分别创建两个 `N×N` 的随机 FP32 矩阵
2. 分别在 CPU 和 NPU 上执行矩阵乘法，用 `time.time()` 计时
3. 验证 CPU 和 NPU 结果一致（`torch.allclose`，容差 `atol=1e-1`）
4. 打印每种规模下 CPU 和 NPU 的耗时及加速比

**API 速查**：

| 操作 | API | 示例 |
|:---|:---|:---|
| 创建随机矩阵 | `torch.randn(N, N, dtype=...)` | `a = torch.randn(512, 512, dtype=torch.float32)` |
| 搬到 NPU | `tensor.npu()` | `a_npu = a.npu()` |
| 矩阵乘法 | `torch.matmul(a, b)` 或 `a @ b` | `c = torch.matmul(a_npu, b_npu)` |
| NPU 同步 | `torch.npu.synchronize()` | `torch.npu.synchronize()` |
| 搬回 CPU | `tensor.cpu()` | `c_cpu = c_npu.cpu()` |
| 验证（带容差） | `torch.allclose(a, b, atol=...)` | `torch.allclose(c_cpu, c_npu.cpu(), atol=1e-1)` |
| 计时 | `time.time()` | `t0 = time.time(); ...; elapsed = time.time() - t0` |

**提示**：
- NPU 计算是异步的，计时前需要 `torch.npu.synchronize()` 确保计算完成
- 注意：NPU 首次执行会有编译开销，如果想测纯计算性能可以先做一次 warmup

In [ ]:
import time

# ========== 第 2 题：CPU vs NPU 矩阵乘法性能对比 ==========
# TODO: 在下方完成你的代码

sizes = [128, 512, 2048]

print(f"{'规模':>8} | {'CPU 耗时':>12} | {'NPU 耗时':>12} | {'加速比':>8} | {'结果一致':>8}")
print("-" * 65)

for N in sizes:
    # Step 1: 创建两个 N×N 随机矩阵（CPU 上）
    a_cpu = 
    b_cpu = 

    # Step 2: CPU 矩阵乘法计时
    # 提示: c_cpu = torch.matmul(a_cpu, b_cpu)
    

    # Step 3: NPU 矩阵乘法计时（别忘了 synchronize）
    # 提示: a_npu = a_cpu.npu() → c_npu = torch.matmul(...) → synchronize
    

    # Step 4: 验证结果一致
    # 提示: consistent = torch.allclose(c_cpu, c_npu.cpu(), atol=1e-1)
    

    # 打印结果（已提供）
    speedup = t_cpu / t_npu
    tag = '✓' if consistent else '✗'
    print(f"{N:>8} | {t_cpu*1000:>10.2f}ms | {t_npu*1000:>10.2f}ms | {speedup:>7.1f}x | {tag:>8}")


#### 思考与讨论

**你测出来的加速比是多少？** 不同规模（128、512、2048）下加速比差异很大，想想为什么。

**这个加速比还能再提升吗？**

目前我们用的是 FP32（单精度浮点数）。但 NPU 的 Cube 计算单元针对 **FP16**（半精度）优化，吞吐量约为 FP32 的 3 倍。

> 💡 **行业趋势**：当前大模型时代，训练一般用 FP16 甚至 FP8，推理一般会做量化（INT8/INT4），最新硬件已在探索 FP4。低精度是提升算力的关键方向之一。

下面用 FP16 试试，感受精度与速度的权衡：

In [ ]:
# ========== 选做：FP32 vs FP16 对比 ==========
N = 2048
reps = 20

a = torch.randn(N, N, dtype=torch.float32)
b = torch.randn(N, N, dtype=torch.float32)

# CPU FP32 基准
t0 = time.time()
for _ in range(reps): c_cpu = torch.matmul(a, b)
t_cpu = (time.time() - t0) / reps * 1000

# NPU FP32
a32, b32 = a.npu(), b.npu()
# Warmup: NPU 首次执行会编译 Kernel（JIT），耗时较长，先跑几次排除编译开销
for _ in range(3): _ = torch.matmul(a32, b32)
torch.npu.synchronize()
t0 = time.time()
for _ in range(reps): c_npu32 = torch.matmul(a32, b32)
torch.npu.synchronize()
t_npu32 = (time.time() - t0) / reps * 1000

# NPU FP16
a16, b16 = a.half().npu(), b.half().npu()
for _ in range(3): _ = torch.matmul(a16, b16)  # 同样先 warmup
torch.npu.synchronize()
t0 = time.time()
for _ in range(reps): c_npu16 = torch.matmul(a16, b16)
torch.npu.synchronize()
t_npu16 = (time.time() - t0) / reps * 1000

# 对比
print(f"{'精度':>8} | {'NPU 耗时':>10} | {'加速比':>10} | {'与CPU最大差异':>14}")
print("-" * 55)
print(f"{'FP32':>8} | {t_npu32:>8.2f}ms | {t_cpu/t_npu32:>9.1f}x | {(c_cpu - c_npu32.cpu()).abs().max().item():>14.4f}")
print(f"{'FP16':>8} | {t_npu16:>8.2f}ms | {t_cpu/t_npu16:>9.1f}x | {(c_cpu.float() - c_npu16.cpu().float()).abs().max().item():>14.4f}")
print(f"\nFP16 比 FP32 快 {t_npu32/t_npu16:.1f}x，但精度有所下降")


#### 选做：FP32 vs FP16 精度与速度对比

NPU 的 Cube 单元针对 **FP16** 优化，吞吐量约为 FP32 的 3 倍。但 FP16 精度较低（尾数 10 位 vs FP32 的 23 位）。

用 N=2048 矩阵乘法对比：CPU FP32 为基准，NPU 分别用 FP32 和 FP16 计算，观察速度提升与精度损失。

> **思考**：精度损失对 AI 模型训练/推理是否可接受？什么场景下必须用 FP32？

In [ ]:
# ========== 选做：FP32 vs FP16 对比 ==========
N = 2048
reps = 20

a = torch.randn(N, N, dtype=torch.float32)
b = torch.randn(N, N, dtype=torch.float32)

# CPU FP32 基准
t0 = time.time()
for _ in range(reps): c_cpu = torch.matmul(a, b)
t_cpu = (time.time() - t0) / reps * 1000

# NPU FP32
a32, b32 = a.npu(), b.npu()
for _ in range(3): _ = torch.matmul(a32, b32)
torch.npu.synchronize()
t0 = time.time()
for _ in range(reps): c_npu32 = torch.matmul(a32, b32)
torch.npu.synchronize()
t_npu32 = (time.time() - t0) / reps * 1000

# NPU FP16
a16, b16 = a.half().npu(), b.half().npu()
for _ in range(3): _ = torch.matmul(a16, b16)
torch.npu.synchronize()
t0 = time.time()
for _ in range(reps): c_npu16 = torch.matmul(a16, b16)
torch.npu.synchronize()
t_npu16 = (time.time() - t0) / reps * 1000

# 对比
print(f"{'精度':>8} | {'NPU 耗时':>10} | {'加速比':>10} | {'与CPU最大差异':>14}")
print("-" * 55)
print(f"{'FP32':>8} | {t_npu32:>8.2f}ms | {t_cpu/t_npu32:>9.1f}x | {(c_cpu - c_npu32.cpu()).abs().max().item():>14.4f}")
print(f"{'FP16':>8} | {t_npu16:>8.2f}ms | {t_cpu/t_npu16:>9.1f}x | {(c_cpu.float() - c_npu16.cpu().float()).abs().max().item():>14.4f}")
print(f"\nFP16 比 FP32 快 {t_npu32/t_npu16:.1f}x，但精度有所下降")


### 第 3 题（挑战 ★★★）：Batch 矩阵乘法 vs 循环单次矩阵乘法

**背景**：通过 `torch_npu` 调用的每个 API 都对应一个 CANN 算子，即一次 NPU Kernel 启动。`torch.bmm` 一次调用完成 B 个矩阵乘法，而循环 `torch.matmul` 需要 B 次独立调用——每次调用都有 Kernel 启动开销和 Python 调度开销。随着 B 增大，差异会越来越显著。

**要求**：
1. 对 batch 大小 `B = [8, 32, 128]`，创建形状为 `(B, 256, 256)` 的随机 FP32 矩阵对，搬到 NPU
2. **Batch 方式**：用 `torch.bmm(a, b)` 一次完成 B 个矩阵乘法，计时
3. **循环方式**：用 `for i in range(B): torch.matmul(a[i], b[i])` 逐个计算，计时
4. 验证两种方式结果一致（`torch.allclose`，`atol=1e-1`）
5. 打印每种 batch 大小下两种方式的耗时及加速比，观察 B 增大时差异如何变化

**思考题**（选做）：
- 为什么 B 越大，循环方式越慢？开销来自哪里？
- 什么情况下循环方式的劣势不明显？（提示：矩阵本身很大时）
- 在模型开发中，如何避免不必要的循环调用？

> 📌 **后续课程将深入解释**：Kernel 启动开销的底层原因、NPU 任务调度器（TS）的工作机制、以及如何通过算子融合进一步减少调用次数，将在 Ascend C 算子开发系列课程中详细讲解。

**API 速查**：

| 操作 | API | 示例 |
|:---|:---|:---|
| 创建 batch 张量并搬到 NPU | `torch.randn(B, M, K, dtype=...).npu()` | `a = torch.randn(32, 256, 256, dtype=torch.float32).npu()` |
| Batch 矩阵乘法 | `torch.bmm(a, b)` | `c = torch.bmm(a, b)`  # a:(B,M,K) b:(B,K,N) → c:(B,M,N) |
| 单次矩阵乘法 | `torch.matmul(a[i], b[i])` | `c_i = torch.matmul(a[0], b[0])` |
| NPU 同步 | `torch.npu.synchronize()` | `torch.npu.synchronize()` |
| 验证（带容差） | `torch.allclose(a, b, atol=...)` | `torch.allclose(c_bmm, c_loop, atol=1e-1)` |
| 计时 | `time.time()` | `t0 = time.time(); ...; elapsed = (time.time()-t0)/reps*1000` |

In [ ]:
# ========== 第 3 题：Batch 矩阵乘法 vs 循环单次矩阵乘法 ==========
# TODO: 在下方完成你的代码

batch_sizes = [8, 32, 128]
M, K, N = 256, 256, 256
reps = 20

print(f"{'B':>6} | {'bmm 耗时':>12} | {'循环 耗时':>12} | {'加速比':>8} | {'结果一致':>8}")
print("-" * 65)

for B in batch_sizes:
    # Step 1: 创建 (B, M, K) 和 (B, K, N) 随机矩阵，搬到 NPU
    # 提示: a = torch.randn(B, M, K, dtype=torch.float32).npu()
    

    # Step 2: Warmup
    

    # Step 3: torch.bmm 计时
    # 提示: c_bmm = torch.bmm(a, b)
    

    # Step 4: 循环 torch.matmul 计时
    # 提示: c_loop = torch.stack([torch.matmul(a[i], b[i]) for i in range(B)])
    

    # Step 5: 验证结果一致
    # 提示: consistent = torch.allclose(c_bmm, c_loop, atol=1e-1)
    

    # 打印结果（已提供）
    speedup = t_loop / t_bmm
    tag = '✓' if consistent else '✗'
    print(f"{B:>6} | {t_bmm:>10.2f}ms | {t_loop:>10.2f}ms | {speedup:>7.1f}x | {tag:>8}")


### 第 4 题（挑战 ★★★）：在 NPU 上加速图像高斯模糊

**背景**：高斯模糊是计算机视觉中最基础的操作之一，底层是**卷积运算**（Convolution），正是 NPU 的 Cube 计算单元最擅长的场景。

**要求**：
1. 在 CPU 上执行卷积并计时（`F.conv2d`，`padding=2`，`groups=3`）
2. 将图像和卷积核搬到 NPU，执行卷积，结果搬回 CPU，计时（别忘了 `synchronize`）
3. 观察加速比，思考：为什么单张图片的加速比不高？

**提示**：
- NPU 计算是异步的，计时前后都需要 `torch.npu.synchronize()`
- 图片加载、高斯核构造、可视化代码已提供，你只需填写 `# TODO` 标注的运算部分

In [ ]:
import torch, torch_npu, torch.nn.functional as F
from PIL import Image
import numpy as np, matplotlib.pyplot as plt, time

# ========== 第 4 题：在 NPU 上加速图像高斯模糊 ==========

# Step 1: 加载图片并转为张量 [1, 3, H, W]（已提供）
img = Image.open('./images/lena.jpg').convert('RGB')
img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).unsqueeze(0).float() / 255.0
print(f"图片张量形状: {img_tensor.shape}")

# 显示输入图片
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.title('Input: Lena (512x512)')
plt.axis('off')
plt.show()

# Step 2: 构造 5×5 高斯卷积核（已提供）
size, sigma = 5, 1.0
coords = torch.arange(size, dtype=torch.float32) - size // 2
g1d = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
kernel2d = g1d.unsqueeze(1) * g1d.unsqueeze(0)
kernel2d = kernel2d / kernel2d.sum()
weight = kernel2d.unsqueeze(0).unsqueeze(0).repeat(3, 1, 1, 1)  # [3, 1, 5, 5]

# TODO: Step 3: CPU 卷积计时（1 行代码）
# blur_cpu = F.conv2d(...)



# TODO: Step 4: NPU 卷积计时（搬到 NPU → synchronize → 计时 → conv2d → synchronize → 搬回 CPU）


# Step 5: 打印耗时与加速比（已提供）
print(f"CPU 卷积耗时: {cpu_time*1000:.2f} ms")
print(f"NPU 卷积耗时: {npu_time*1000:.2f} ms")
print(f"加速比: {cpu_time/npu_time:.1f}x")

# Step 6: 显示输出图片（已提供）
blur_img = Image.fromarray((result.squeeze(0).permute(1, 2, 0).clamp(0, 1).numpy() * 255).astype(np.uint8))
plt.figure(figsize=(6, 5))
plt.imshow(blur_img)
plt.title(f'Gaussian Blur (NPU, {npu_time*1000:.1f} ms)')
plt.axis('off')
plt.show()


---
## 二、验证与答案

完成全部四道实践题后，运行下方代码查看参考答案与批改结果。

In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / 'quick_start' / 'cann_basics' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find quick_start/cann_basics/answer')
from grade_04 import grade
grade(globals())

---
## 三、下一步学习建议

完成本课程后，你已经具备了 CANN 和昇腾 NPU 的基础认知。接下来推荐以下方向：

| 方向 | 链接 | 适合人群 |
|:---|:---|:---|
| 体验自定义算子开发 | [第一个自定义算子](../first_custom_operator/first_custom_operator.ipynb) | 想了解算子是怎么写的 |
| 体验算子 API 调用 | [第一个算子 API 调用](../first_operator_api_call/first_operator_api_call.ipynb) | 想了解如何调用 CANN 内置算子 |
| 系统学习算子开发 | [Ascend C 算子开发系列](../../tutorials/ascendc_operator_development_light) | 想掌握底层编程能力 |
| 大模型推理实战 | [Qwen3-8B 推理](../../tutorials/llm_inference/qwen3_8b/02_baseline_inference.ipynb) | 想跑通真实大模型 |

> 💡 **学习路径建议**：先体验`第一个自定义算子`感知算子开发全流程 → 再系统学习 Ascend C 算子开发系列 → 最后结合大模型推理/训练深入实践。